# Preprocessing – Project Notebook

Use this notebook for carrying out the analyses from the workshop notebook on your own subreddit data.

NB. Make sure you've installed all required packages (code is in the Lesson file).

### Icons Used in This Notebook
💭 **Reflection**: Reflecting on ethical implications, biases, and social impact in data science.<br>

> **Data transparency note**: The analyses in this notebook operate on text written by real people in the community you collected. The patterns these methods surface reflect that community's discourse — not universal truths about language or the people writing. Keep this context in mind when interpreting results.

## Reading the Data

Put your data in the `data` folder of this repo and replace `YOUR_FILE.csv` below with the name of your file.

In [ ]:
# Import the pandas package
import pandas as pd 

# Replace this with your own file!
df = pd.read_csv('../../data/YOUR_FILE.csv')

Check out the shape, first rows, and columns.

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
# This allows you to quickly see which columns you have
list(df)

## 💭 Reflection

Take some time to look at the reddit community you have chosen. If it no longer exists, look at some of the posts in your DataFrame. Think about the following questions:

- What kinds of norms and values does this community seem to be organized around?
- Does the community include a FAQ or Wiki page that explain rules for interaction among members? What are these rules?
- What are the most popular posts of all time? Do they have something in common?
- Are there dissenting voices when it comes to these norms? How do others respond to them?

## Reviewing Posts

Plot post length:

In [ ]:
import matplotlib.pyplot as plt

# Compute text length if your data doesn't have a 'textlen' column
# Change 'selftext' to 'body' if you are working with a comments file!
if 'textlen' not in df.columns:
    df['textlen'] = df['selftext'].str.split().str.len()

plt.figure(figsize=(8,5))
df['textlen'].dropna().hist(bins=30)
plt.title("Distribution of Text Lengths in Subreddit Posts")
plt.xlabel("Text Length")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("outputs_project/text_length_distribution.png", dpi=300)  # exportable
plt.show()

Plot comment count vs. score:

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df['num_comments'], df['score'], alpha=0.5)
plt.title("Comments vs. Score")
plt.xlabel("Number of Comments")
plt.ylabel("Post Score")
plt.tight_layout()
plt.savefig("outputs_project/comments_vs_score.png", dpi=300)
plt.show()

In [ ]:
# Sort by score to find top outliers
df.sort_values(by='score', ascending=False).head(3)

## Removing columns and rows

NOTE: If you get a `KeyError` in the cell below, just skip the cell. It means your dataset does not include the columns we are dropping here.

In [ ]:
# Drop some columns
df = df.drop(['self', 'url', 'subreddit', 'augmented_at', 'augmented_count'], axis=1)

NOTE: If you are preprocessing a **comments** file, the `selftext` column below should be called `body`! Make sure to replace this in the code below or you'll get a `KeyError`.

In [ ]:
# Select all rows that don't have '[removed]' or '[deleted]' in the post's text.
# We use str.contains() rather than isin() to also catch posts where these strings
# appear as part of a longer partially-removed text.
mask_removed = df['selftext'].str.contains(r'\[removed\]|\[deleted\]', na=False)
df = df.loc[~mask_removed, :]

# Select all rows that have >3 characters in selftext
df = df.loc[df['selftext'].str.len() > 3]

df.shape

In [ ]:
# Drop null values in selftext
df = df.dropna(subset=['selftext'])
df.shape

## Cleaning Reddit Markdown

Reddit posts contain markdown formatting artifacts that should be removed before analysis: hyperlinks, bold/italic markers, subreddit and username mentions, and edit notices. The function below handles these common patterns.

This kind of cleaning is **interpretive**: each decision about what to strip shapes what your model will "see". For instance, removing `r/` mentions means your topic model won't surface community references as signals. Removing `EDIT:` prefixes means edits get treated the same as original text. Think carefully about whether any of these signals matter for your research question.

In [ ]:
import re

def clean_reddit_markdown(text):
    """Remove common Reddit markdown artifacts before NLP processing."""
    # Remove hyperlinks: [link text](url) → link text
    text = re.sub(r'\[([^\]]+)\]\(https?://[^\)]+\)', r'\1', text)
    # Remove bare URLs
    text = re.sub(r'https?://\S+', '', text)
    # Remove bold/italic markers (**text** or *text* or __text__)
    text = re.sub(r'\*{1,2}([^\*]+)\*{1,2}', r'\1', text)
    text = re.sub(r'_{1,2}([^_]+)_{1,2}', r'\1', text)
    # Remove subreddit and username mentions
    text = re.sub(r'r/\w+', '', text)
    text = re.sub(r'u/\w+', '', text)
    # Remove EDIT/UPDATE prefixes (common in Reddit posts)
    text = re.sub(r'\bEDIT\b.*?(\n|$)', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\bUPDATE\b.*?(\n|$)', ' ', text, flags=re.IGNORECASE)
    # Collapse extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Change 'selftext' to 'body' if you are working with a comments file!
df['selftext'] = df['selftext'].apply(clean_reddit_markdown)

## Preprocessing Data with Spacy

In [ ]:
# Install all required packages for this notebook
#%pip install gensim spacy pandas tqdm

# Install spacy English model
import spacy
try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    from spacy.cli import download
    download('en_core_web_sm')
    nlp = spacy.load('en_core_web_sm')

NOTE: The following cell includes the preprocessing steps we do with spaCy. If you want to add or change any steps, this is the function you should alter.

In [ ]:
def process_text(text):
    """Function to process a single text string."""

    # Replace newlines with spaces so words at line boundaries aren't merged
    text = text.replace('\n', ' ')
    parsed = nlp(text, disable=["tok2vec", "ner"])

    # Gather lowercased, lemmatized tokens that are not punctuation, space, or digit
    # Note: spaCy v3 no longer uses '-PRON-'; lemma_ returns the actual token form for pronouns
    tokens = [
        token.lemma_.lower()
        for token in parsed 
        if not (token.is_punct or token.is_space or token.is_digit)
    ]

    # Remove specific lemmatizations
    tokens = [
        lemma
        for lemma in tokens
        if not lemma in ["'s",  "’s", "'"]
    ]

    # Remove stop words
    tokens = [
        token 
        for token in tokens 
        if token not in spacy.lang.en.stop_words.STOP_WORDS
    ]

    return ' '.join(tokens)

from tqdm import tqdm

def preprocess(df, text_col='selftext'):
    """Preprocessing function to apply to a dataframe."""
    # Enable tqdm for pandas
    tqdm.pandas(desc="Processing text")
    
    df['pp_text'] = df[text_col].progress_apply(process_text)
    return df

NOTE: In the following cell, change the `text_col` parameter from `selftext` to `body` if you are preprocessing a comments file!

In [ ]:
# This may take a while
df = preprocess(df, text_col='selftext')
df.reset_index(drop=True, inplace=True)

## Phrase modeling

In [ ]:
from gensim.models.phrases import Phrases, Phraser
from tqdm import tqdm

# Create bigram and trigram models
print("Tokenizing documents...")
tokens = [doc.split(" ") for doc in tqdm(df['pp_text'], desc="Tokenizing")]

print("Training bigram model...")
bigram = Phrases(tokens, min_count=10, threshold=100)

print("Training trigram model...")
trigram = Phrases(bigram[tokens], min_count=10, threshold=50)  

bigram_phraser = Phraser(bigram)
trigram_phraser = Phraser(trigram)

# Form trigrams with progress
print("Forming trigrams...")
df['pp_text'] = [' '.join(trigram_phraser[bigram_phraser[doc]]) 
                 for doc in tqdm(tokens, desc="Creating trigrams")]

In [ ]:
# Check out the first post
df['pp_text'][0]

Check how many bigrams were identified by the parser.

In [ ]:
len(bigram_phraser.phrasegrams.keys())

Print the first few bigrams identified in the model to check if they seem  appropriate. If not, you can play around with the parameters of the bigram model to adjust the sensitivity of the model (the values for `min_count` and `threshold` above).

In [ ]:
list(bigram_phraser.phrasegrams.keys())[:10]

In [ ]:
# Look at trigrams
[trigram for trigram in list(trigram_phraser.phrasegrams.keys()) if trigram.count('_') == 2]

## 💭 Reflection: The Hermeneutics of Cleaning

Every preprocessing decision is an interpretive act. When you strip stopwords, you assume that grammar and function words carry no meaning worth preserving — but in many online communities, phrases like "I just" or "not even" may carry significant social weight.

Consider these questions:

- **What did you lose by lowercasing?** Capitalization can signal emphasis ("I was NOT wrong"), shouting, or proper names.
- **What did removing stopwords cost you?** Negations ("did not", "never asked") often hinge on function words that are stripped out.
- **What did the Reddit markdown cleaner erase?** Links, edit notices, and subreddit mentions can themselves be data — they reveal how posters situate themselves in the platform's culture.
- **Who chose what counts as "noise"?** The `STOP_WORDS` list in spaCy was designed for general English text. How might that list perform differently on the kind of language used in your community?

🔔 **Question**: Pick one specific preprocessing step you applied above and argue *for* removing it in a research context where keeping that information would matter. What research question would change if you kept it?

## Saving data

Let's save this cleaned-up and preprocessed dataframe in a new CSV.

Change this to a name of your own! However, make sure to **not** give it the same name as your original .CSV file and overwrite it. It's good practice to keep that original file intact.

In [ ]:
df.to_csv('../../data/YOUR_FILE_PP.csv', index=False)

Make sure to swap out `YOUR_FILE` for something relevant – e.g. `my_subreddit_comments_PP` or `my_subreddit_submissions_PP`. Keeping the `_PP` ("preprocessed") suffix is a good idea: next week's project notebook expects to load a preprocessed file.

In case you want to save just the text in binary format, you can use pickle. Make sure to change the file name, again.

In [ ]:
import pickle

with open('../../data/YOUR_FILE_PP.pickle', 'wb') as f:
    # Save (or "dump") the object into the file
    pickle.dump(df['pp_text'].tolist(), f)